# Fusionner un checkpoint hybride, et le passer dans le pipeline

Prend un checkpoint d'entrainement (une snapshot de phase 2, par exemple), refond les LoRA
dans le backbone pour en faire un modele d'inference, et le fait tourner sur les vrais tickets
labellises, ceux-la memes que le reste du projet utilise pour mesurer.

A lancer sur le kernel `.venv` du projet : la fusion comme l'inference ont besoin des
dependances d'entrainement, qui ne sont pas dans le Python systeme. La cellule suivante
previent si on s'est trompe d'interpreteur.

## Config

In [ ]:
# Le checkpoint a fusionner, rapatrie depuis Kaggle. Si le chemin est faux, la cellule
# de fusion liste ce qui existe a cote.
CHECKPOINT = "checkpoints/phase2_epoch04_loss0.3076.pt"

# Ou ecrire le modele fusionne. "" = a cote du checkpoint.
MERGED_OUT = ""

# Les vrais tickets labellises, les memes que scripts/evaluate.py.
# Laisser vide pour prendre les emplacements par defaut du repo.
IMAGES_DIR = ""
LABELS_DIR = ""
SPLIT = "test"            # test, val, train, ou vide pour tout
REQUIRE_REVIEWED = False  # nos labels sont des pseudo-labels, pas encore relus a la main.
                          # A passer a True une fois qu'ils le seront.
MAX_IMAGES = 0            # 0 = toutes les images du split

## Trouver les paquets, et verifier l'interpreteur

Localise `dev_ocr/vlm_training`, met `receipt_ocr` et `receipt_vlm` sur le path, et rale si
le kernel n'est pas le bon.

In [ ]:
import os, sys
from pathlib import Path

def _find_train_pkg() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "scripts" / "export_checkpoint.py").is_file():
            return base
        cand = base / "dev_ocr" / "vlm_training"
        if (cand / "scripts" / "export_checkpoint.py").is_file():
            return cand
    raise RuntimeError("Couldn't locate dev_ocr/vlm_training -- set TRAIN_PKG manually")

TRAIN_PKG = _find_train_pkg()
DEV_OCR = TRAIN_PKG.parent
for p in (str(DEV_OCR / "src"), str(TRAIN_PKG)):
    if p not in sys.path:
        sys.path.insert(0, p)

# Le venv du projet a les bonnes deps. Les sous-processus l'utilisent, et on rale si le
# kernel courant n'est pas celui-la.
_venv = TRAIN_PKG / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
VENV_PY = str(_venv) if _venv.exists() else sys.executable
on_venv = (not _venv.exists()) or Path(sys.executable).resolve() == _venv.resolve()
if not on_venv:
        print("This kernel is NOT the project venv:")
    print("   kernel :", sys.executable)
    print("   venv   :", VENV_PY)
    print("Cell 2 will still merge (it uses the venv), but cell 3 inference runs IN this kernel")
    print("and will fail on missing/old deps. Switch the kernel to .venv, then Run All.")
    
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Train package:", TRAIN_PKG)
print("Device       :", DEVICE,
      "-", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only (slow generation)")

## Fusionner le checkpoint

Appelle `scripts/export_checkpoint.py`, qui replie les adaptateurs LoRA dans le backbone gele
et ecrit un seul `.pt`. La sortie du script est affichee telle quelle : un echec doit se voir,
pas se cacher derriere un `CalledProcessError` opaque.

In [ ]:
import subprocess
from pathlib import Path

ckpt = Path(CHECKPOINT)
if not ckpt.is_file():
    avail = sorted(p.name for p in ckpt.parent.glob("*.pt")) if ckpt.parent.is_dir() else []
    raise FileNotFoundError(
        f"CHECKPOINT not found: {ckpt.resolve()}\n"
        f"Available in {ckpt.parent}/: {avail or 'none -- download one from your Kaggle dataset'}"
    )
merged = Path(MERGED_OUT) if MERGED_OUT else ckpt.with_name("receipt_vlm_500m_merged.pt")

proc = subprocess.run(
    [VENV_PY, str(TRAIN_PKG / "scripts" / "export_checkpoint.py"),
     "--checkpoint", str(ckpt), "--output", str(merged)],
    capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"export_checkpoint.py failed (exit {proc.returncode}) -- see error above")

MERGED = str(merged.resolve())
print("Merged ->", MERGED, f"({merged.stat().st_size/1e6:.0f} MB)")
print("Built from:", ckpt.name)

## Faire tourner le pipeline sur les vrais tickets

Charge les memes tickets labellises que `scripts/evaluate.py`, les passe dans le pipeline
complet, et affiche chaque prediction a cote de son label pour qu'on puisse juger a l'oeil.

Si on obtient 0 echantillon, c'est que les labels ne sont pas encore relus a la main : laisser
`REQUIRE_REVIEWED=False`.

In [ ]:
import os, json, time
from pathlib import Path

os.environ.update({
    "RECEIPT_OCR_BACKEND":    "vlm",
    "RECEIPT_VLM_MODEL":      "receipt-vlm-500m",
    "RECEIPT_VLM_MODE":       "json",
    "RECEIPT_VLM_MODEL_PATH": MERGED,
})

from receipt_ocr import extract_receipt                       # le pipeline complet
from receipt_vlm.data.real_photos import load_real_samples    # le meme chargeur que evaluate.py
from receipt_vlm.data.schema import ticket_from_dict

images_dir = Path(IMAGES_DIR) if IMAGES_DIR else DEV_OCR / "data" / "raw" / "images_tickets_caisse"
labels_dir = Path(LABELS_DIR) if LABELS_DIR else TRAIN_PKG / "data" / "real_labels"
samples = load_real_samples(images_dir, labels_dir, split=(SPLIT or None),
                            require_reviewed=REQUIRE_REVIEWED)
if not samples:
    raise SystemExit(
        f"No labelled samples for split={SPLIT!r} (require_reviewed={REQUIRE_REVIEWED}) "
        f"under {labels_dir}. If the labels aren't hand-reviewed, set REQUIRE_REVIEWED=False."
    )
if MAX_IMAGES:
    samples = samples[:MAX_IMAGES]
print(f"Testing on {len(samples)} labelled '{SPLIT or 'all'}' receipts from {images_dir}\n")

predictions, golds, n_valid = [], [], 0
for s in samples:
    t = time.time()
    try:
        out = extract_receipt(str(s.image))
        pred = ticket_from_dict(out)
        n_valid += 1
    except Exception as e:
        out = {"error": f"{type(e).__name__}: {e}"}
        pred = ticket_from_dict({})
    predictions.append(pred)
    golds.append(s.ticket)
    print(f"=== {Path(s.image).name}  ({time.time()-t:.1f}s) ===")
    print("  predicted:", json.dumps(out, ensure_ascii=False)[:700])
    print("  gold     :", json.dumps(s.ticket.to_dict(), ensure_ascii=False)[:700], "\n")

print(f"Valid JSON: {n_valid}/{len(samples)}")

## Les metriques, face a la verite terrain

Les memes metriques que `scripts/evaluate.py`, mais calculees sur la sortie du pipeline, donc
sur le chemin reellement deploye et pas sur le modele nu.

Le jeu labellise est minuscule : ces chiffres donnent une direction, pas une mesure.

In [ ]:
from receipt_vlm.utils.metrics import evaluate_tickets

metrics = evaluate_tickets(predictions, golds)
TARGETS = [
    ("field_f1",       "Field F1",         "> 0.85"),
    ("product_recall", "Product recall",   "> 0.90"),
    ("price_mae",      "Price MAE (EUR)",  "< 0.05"),
    ("date_accuracy",  "Date exact match", "> 0.90"),
    ("anls",           "ANLS",             "> 0.80"),
]
print(f"{'Metric':22} {'Target':10} {'receipt-vlm-500m'}")
print("-" * 50)
for key, label, target in TARGETS:
    if key in metrics:
        print(f"{label:22} {target:10} {metrics[key]:.3f}")
print(f"\nn = {len(predictions)} labelled receipts  |  checkpoint: {Path(CHECKPOINT).name}")

## Le deployer

Le modele fusionne est un seul `.pt`. C'est le worker `workers/ocr-vlm-receipt` qui le sert :
il le telecharge depuis GCS au demarrage et le passe a son backend.

Attention : ce modele n'est PAS branchable sur le `receipt_ocr` de `dev_ocr`, dont le registre
ne connait que Moondream et Groq. Il ne vit que dans la lib de deploiement.